In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from langdetect import detect
import re
import emoji
from pathlib import Path
import os

In [ ]:
# ================= PROJECT ROOT & DATA PATHS =================
try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()  # fallback if running interactively

DEFAULT_DATA_DIR = Path(r"C:/Users/DELL/Documents/project_data/output")
DATA_DIR = Path(os.getenv("DATA_DIR", DEFAULT_DATA_DIR))

POSTS_PATH = DATA_DIR / "safaricom_posts.csv"
COMMENTS_PATH = DATA_DIR / "safaricom_comments22.csv"

# ================= FILE EXISTENCE CHECK =================
print("Using DATA_DIR:", DATA_DIR)
if not POSTS_PATH.exists():
    raise FileNotFoundError(f"Posts file not found: {POSTS_PATH}")
if not COMMENTS_PATH.exists():
    raise FileNotFoundError(f"Comments file not found: {COMMENTS_PATH}")

# ================= LOAD CSVs =================
posts = pd.read_csv(POSTS_PATH)
comments = pd.read_csv(COMMENTS_PATH)


comments_sent = comments[[ 'text']].dropna()

#================== DATA INSPECTION =================
print("\n=== BEFORE PREPROCESSING ===")
print(f"Posts shape: {posts.shape}")
print(f"Comments shape: {comments.shape}")
print("\nPosts columns:", posts.columns.tolist())
print("Comments columns:", comments.columns.tolist())

print("\n------ For Sentiment Analysis ------")
print(f"Comments shape: {comments_sent.shape}")


Using DATA_DIR: C:\Users\DELL\Documents\project_data\output

=== BEFORE PREPROCESSING ===
Posts shape: (1948, 11)
Comments shape: (286376, 5)

Posts columns: ['msg_id', 'date_utc', 'text', 'views', 'forwards', 'replies', 'has_media', 'media_type', 'sender_id', 'reply_to_msg_id', 'num_comments']
Comments columns: ['post_id', 'comment_id', 'date_utc', 'text', 'sender_id']

------ After concatenation ------
Concatenated data shape: (281445, 2)

Data columns: ['id', 'text']


In [ ]:
# ================= FILL MISSING VALUES =================
posts['media_type'] = posts['media_type'].fillna('Unknown')
posts['text'] = posts['text'].fillna('<media_only>')
posts['reply_to_msg_id'] = posts['reply_to_msg_id'].fillna('None')

comments['text'] = comments['text'].fillna('<deleted>')

In [ ]:
# ==============================
# COMPLETE EMOJI DETECTION + CONVERSION PIPELINE (ONE CELL)
# ==============================

import pandas as pd
import regex as re
import emoji
import matplotlib.pyplot as plt

# ------------------------------
# 1. Unicode patterns
# ------------------------------
AMHARIC_RE = re.compile(r'[\u1200-\u137F]')
ENGLISH_RE = re.compile(r'[A-Za-z]')
EMOJI_RE = emoji.get_emoji_regexp()

# ------------------------------
# 2. Classification function
# ------------------------------
def classify_emoji_text(text: str) -> str:
    if not isinstance(text, str) or text.strip() == "":
        return "empty"

    text = text.strip()

    has_emoji = bool(EMOJI_RE.search(text))
    has_amharic = bool(AMHARIC_RE.search(text))
    has_english = bool(ENGLISH_RE.search(text))

    if has_emoji and not (has_amharic or has_english):
        return "emoji_only"
    elif has_emoji and (has_amharic or has_english):
        return "emoji_with_text"
    elif has_amharic or has_english:
        return "text_only"
    else:
        return "other"

# ------------------------------
# 3. Emoji → text conversion
# ------------------------------
def demojize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = text.replace("_", " ").replace(":", "")
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ------------------------------
# 4. Final cleaned modeling text
# ------------------------------
def final_clean_text(text: str) -> str:
    text = demojize_text(text)
    text = re.sub(r'http\S+|www\S+|@\w+', '', text)
    text = re.sub(r'[^\w\s\u1200-\u137F]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ------------------------------
# 5. Apply to comments dataframe
# ------------------------------
comments['emoji_text_type'] = comments['text'].apply(classify_emoji_text)
comments['model_text'] = comments['text'].apply(final_clean_text)

# ------------------------------
# 6. Statistics
# ------------------------------
counts = comments['emoji_text_type'].value_counts()
percentages = counts / len(comments) * 100

print("=== EMOJI / TEXT DISTRIBUTION ===")
for label in counts.index:
    print(f"{label:18}: {counts[label]:10,} ({percentages[label]:6.2f}%)")

print("\nEmoji-only comments:",
      f"{counts.get('emoji_only', 0):,}")

# ------------------------------
# 7. Visualization (warning-free matplotlib only)
# ------------------------------
plt.figure(figsize=(10,6))
plt.bar(counts.index, counts.values)
plt.xticks(rotation=45)
plt.title("Emoji vs Text Comment Distribution")
plt.xlabel("Comment Type")
plt.ylabel("Number of Comments")

# Annotate percentages
for i, v in enumerate(counts.values):
    plt.text(i, v + max(counts.values)*0.01,
             f"{percentages.iloc[i]:.1f}%",
             ha='center')

plt.tight_layout()
plt.show()

# ------------------------------
# 8. Preview transformation quality
# ------------------------------
print("\n=== SAMPLE TRANSFORMATIONS ===")
display(
    comments[['text','model_text','emoji_text_type']]
    .sample(10, random_state=42)
)